## **Cargar CSV Pandas** 🐍
Es una biblioteca de Python de código abierto especializada en la manipulación y el análisis de datos. Su nombre proviene del término en inglés "Panel Data" (datos de panel).

Estructuras de datos principalesSeries: 
* Una matriz unidimensional con etiquetas que puede contener cualquier tipo de datos.
* DataFrame: Una tabla bidimensional con filas y columnas etiquetadas, similar a una hoja de cálculo de Excel o una tabla SQL.

Para qué sirve
* Leer y escribir archivos: Permite importar y exportar datos en formatos como CSV, Excel, JSON y bases de datos SQL.
* Limpieza de datos: Facilita la detección, eliminación o relleno de valores faltantes o nulos.
* Transformación y análisis: Permite filtrar, agrupar, ordenar y combinar grandes volúmenes de información de manera rápida y eficiente.

In [1]:
#pip install pandas
#pip install numpy

import pandas as pd

#importar archivo csv
df = pd.read_csv('dataset_ventas.csv')

# df.head()# 5 primeras filas
# df.tail()# 5 últimas filas
# df.sample(n=5) # 5 filas aleatorias
# df.info() # información general sobre el DataFrame
# df.describe() # información sobre el DataFrame

df.shape # número de filas y columnas

(1000, 7)

### 1. Manejo de Valores faltantes y duplicados

In [2]:
# 1. Identificar valores nulos en el DataFrame
print('--- VALORES NULOS ---')
print(df.isnull().sum())

# 2. Rellenar valores nulos si existieran
df.fillna({'cantidad': df['cantidad'].median(), 'precio_unitario': df['precio_unitario'].median()}, inplace=True)

# 3. Verificar y eliminar valores duplicados
print('--- VALORES DUPLICADOS ---')
duplicados_inicio = df.duplicated().sum()
print(f'Cantidad de filas duplicadas detectadas: {duplicados_inicio}')

df = df.drop_duplicates()
print(f'Filas totales tras eliminar duplicados: {df.shape[0]}')


--- VALORES NULOS ---
fecha              0
ciudad             0
categoria          0
producto           0
cantidad           0
precio_unitario    0
total_venta        0
dtype: int64
--- VALORES DUPLICADOS ---
Cantidad de filas duplicadas detectadas: 0
Filas totales tras eliminar duplicados: 1000


### 2. Codifique variables categóricas usando LabelEncoder y One-Hot Encoding.

In [3]:
from sklearn.preprocessing import LabelEncoder

# Copia del DataFrame para transformaciones
df_encoded = df.copy()

# 1. LabelEncoder para 'producto' y 'ciudad'
le_producto = LabelEncoder()
df_encoded['producto_encoded'] = le_producto.fit_transform(df_encoded['producto'])

le_ciudad = LabelEncoder()
df_encoded['ciudad_encoded'] = le_ciudad.fit_transform(df_encoded['ciudad'])

# 2. One-Hot Encoding para 'categoria'
df_encoded = pd.get_dummies(df_encoded, columns=['categoria'], prefix='cat', dtype=int)

# Extraer mes de la columna fecha y descartar columnas originales de texto
df_encoded['mes'] = pd.to_datetime(df_encoded['fecha']).dt.month
df_encoded.drop(columns=['fecha', 'ciudad', 'producto'], inplace=True)

print('DataFrame tras la codificacion de variables categoricas:')
df_encoded.head()


DataFrame tras la codificacion de variables categoricas:


,cantidad,precio_unitario,total_venta,producto_encoded,ciudad_encoded,cat_Alimentos,cat_Hogar,cat_Papelería,cat_Ropa,cat_Tecnología,mes
0,8,60513,484104,16,4,0,0,0,0,1,11
1,19,112604,2139476,8,4,0,0,1,0,0,3
2,8,134475,1075800,22,4,0,0,0,0,1,1
3,18,111974,2015532,10,4,0,0,1,0,0,11
4,1,200916,200916,13,1,0,0,1,0,0,4


### 3. Aplique normalización y estandarización.

In [4]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Copia para escalado
df_scaled = df_encoded.copy()

# 1. Normalizacion (MinMaxScaler) para 'cantidad' y 'precio_unitario'
min_max_scaler = MinMaxScaler()
cols_to_normalize = ['cantidad', 'precio_unitario']
df_scaled[cols_to_normalize] = min_max_scaler.fit_transform(df_scaled[cols_to_normalize])

# 2. Estandarizacion (StandardScaler) para 'total_venta'
standard_scaler = StandardScaler()
df_scaled[['total_venta']] = standard_scaler.fit_transform(df_scaled[['total_venta']])

print('DataFrame tras aplicar Normalizacion (MinMax) y Estandarizacion (StandardScaler):')
df_scaled.head()


DataFrame tras aplicar Normalizacion (MinMax) y Estandarizacion (StandardScaler):


,cantidad,precio_unitario,total_venta,producto_encoded,ciudad_encoded,cat_Alimentos,cat_Hogar,cat_Papelería,cat_Ropa,cat_Tecnología,mes
0,0.368421,0.235499,-0.775878,16,4,0,0,0,0,1,11
1,0.947368,0.446174,0.702276,8,4,0,0,1,0,0,3
2,0.368421,0.534628,-0.247526,22,4,0,0,0,0,1,1
3,0.894737,0.443626,0.591601,10,4,0,0,1,0,0,11
4,0.000000,0.803339,-1.028749,13,1,0,0,1,0,0,4


### 4. Balancee la variable objetivo usando SMOTE.

In [5]:
from imblearn.over_sampling import SMOTE
import pandas as pd

# Definir las caracteristicas (X) y la variable objetivo (y)
X = df_scaled.drop(columns=['ciudad_encoded'])
y = df_scaled['ciudad_encoded']

print('Distribucion de clases ANTES de SMOTE:')
print(y.value_counts())

# Aplicar SMOTE para balancear la variable objetivo
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print('Distribucion de clases DESPUES de SMOTE:')
print(pd.Series(y_resampled).value_counts())


Distribucion de clases ANTES de SMOTE:
ciudad_encoded
4    204
2    203
1    202
3    197
0    194
Name: count, dtype: int64
Distribucion de clases DESPUES de SMOTE:
ciudad_encoded
4    204
1    204
0    204
2    204
3    204
Name: count, dtype: int64


### 5. Muestre el conjunto de datos final listo para entrenar un modelo de clasificación.

In [6]:
# Combinar caracteristicas (X) y la variable objetivo balanceada (y) en el DataFrame final
df_final = pd.DataFrame(X_resampled, columns=X.columns)
df_final['target_ciudad'] = y_resampled

print(f'Dimensiones finales del dataset: {df_final.shape[0]} filas x {df_final.shape[1]} columnas')
print('Primeras 5 filas del conjunto de datos final listo para entrenamiento:')
df_final.head()


Dimensiones finales del dataset: 1020 filas x 11 columnas
Primeras 5 filas del conjunto de datos final listo para entrenamiento:


,cantidad,precio_unitario,total_venta,producto_encoded,cat_Alimentos,cat_Hogar,cat_Papelería,cat_Ropa,cat_Tecnología,mes,target_ciudad
0,0.368421,0.235499,-0.775878,16,0,0,0,0,1,11,4
1,0.947368,0.446174,0.702276,8,0,0,1,0,0,3,4
2,0.368421,0.534628,-0.247526,22,0,0,0,0,1,1,4
3,0.894737,0.443626,0.591601,10,0,0,1,0,0,11,4
4,0.000000,0.803339,-1.028749,13,0,0,1,0,0,4,1
